# 04. 청크 임베딩

`03_chunking.ipynb`가 만든 청크를 SentenceTransformer로 변환합니다. 첫 실행에서는 모델을 내려받을 수 있고 CPU에서는 수 분 정도 걸릴 수 있습니다. `128`은 입력 토큰 한도이고 `768`은 이 모델의 출력 벡터 차원입니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/RAG/maple_inven_tips_documents_chunked.json').is_file():
            return resolved
    raise FileNotFoundError('03_chunking.ipynb를 먼저 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
SETTINGS_PATH = OUTPUT_ROOT / 'intermediate/pipeline_settings.json'
CHUNKS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_documents_chunked.json'
EMBEDDINGS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings.npy'
MANIFEST_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings_manifest.json'
settings = json.loads(SETTINGS_PATH.read_text(encoding='utf-8'))
MODEL_NAME = settings['model_name']
MAX_TOKENS = settings['max_tokens']
BATCH_SIZE = settings['batch_size']

print('임베딩 모델:', MODEL_NAME)
print('배치 크기:', BATCH_SIZE)

C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임베딩 모델: jhgan/ko-sroberta-multitask
배치 크기: 32


In [2]:
def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def atomic_save_numpy(path, vectors):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('wb', dir=path.parent, suffix='.npy', delete=False) as stream:
            np.save(stream, vectors, allow_pickle=False)
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def build_embedding_text(chunk, tokenizer):
    text = f"{chunk['metadata']['embedding_prefix']}\n\n{chunk['page_content']}"
    token_count = len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
    if token_count > MAX_TOKENS:
        raise ValueError(f"임베딩 입력 토큰 제한 초과: {chunk['id']} ({token_count})")
    return text

def embed_chunks(chunks, model):
    texts = [build_embedding_text(chunk, model.tokenizer) for chunk in chunks]
    vectors = model.encode(
        texts, batch_size=BATCH_SIZE, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=False,
    ).astype(np.float32, copy=False)
    if vectors.ndim != 2 or vectors.shape[0] != len(chunks):
        raise ValueError(f'임베딩 shape 불일치: chunks={len(chunks)}, vectors={vectors.shape}')
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms <= 1e-12):
        raise ValueError('0 벡터는 정규화할 수 없습니다.')
    normalized = (vectors / norms).astype(np.float32, copy=False)
    np.testing.assert_allclose(np.linalg.norm(normalized, axis=1), 1.0, atol=1e-5)
    return normalized

In [3]:
model = SentenceTransformer(MODEL_NAME)
print('모델 입력 토큰 한도:', model.max_seq_length)
print('모델 출력 벡터 차원:', model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2833.80it/s]

모델 입력 토큰 한도: 128
모델 출력 벡터 차원: 768


C:\Users\Playdata\AppData\Local\Temp\ipykernel_27264\853912204.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('모델 출력 벡터 차원:', model.get_sentence_embedding_dimension())


In [4]:
chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
vectors = embed_chunks(chunks, model)
atomic_save_numpy(EMBEDDINGS_PATH, vectors)
manifest = {
    'model_name': MODEL_NAME,
    'chunk_tokens': settings['chunk_tokens'],
    'overlap_tokens': settings['overlap_tokens'],
    'model_max_tokens': MAX_TOKENS,
    'embedding_count': int(vectors.shape[0]),
    'embedding_dimension': int(vectors.shape[1]),
    'dtype': str(vectors.dtype),
    'normalized': True,
    'chunks_sha256': sha256_file(CHUNKS_PATH),
    'embeddings_sha256': sha256_file(EMBEDDINGS_PATH),
    'chunk_ids': [chunk['id'] for chunk in chunks],
    'created_at': datetime.now(timezone.utc).isoformat(),
}
atomic_write_json(MANIFEST_PATH, manifest)
norms = np.linalg.norm(vectors, axis=1)

display({
    '청크 수': len(chunks),
    '임베딩 shape': vectors.shape,
    'dtype': str(vectors.dtype),
    'norm 최솟값': float(norms.min()),
    'norm 최댓값': float(norms.max()),
    '저장 파일': str(EMBEDDINGS_PATH),
})
display(pd.DataFrame({
    'chunk_id': [chunk['id'] for chunk in chunks[:3]],
    'vector_first_8': [vectors[index, :8].tolist() for index in range(3)],
}))

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

Batches:   1%|          | 1/173 [00:05<15:48,  5.51s/it]

Batches:   1%|          | 2/173 [00:10<15:22,  5.40s/it]

Batches:   2%|▏         | 3/173 [00:16<15:20,  5.41s/it]

Batches:   2%|▏         | 4/173 [00:22<15:56,  5.66s/it]

...

Batches:  99%|█████████▉| 172/173 [10:22<00:02,  2.71s/it]

Batches: 100%|██████████| 173/173 [10:22<00:00,  1.95s/it]

Batches: 100%|██████████| 173/173 [10:22<00:00,  3.60s/it]

{'청크 수': 5506,
 '임베딩 shape': (5506, 768),
 'dtype': 'float32',
 'norm 최솟값': 0.9999998807907104,
 'norm 최댓값': 1.0000001192092896,
 '저장 파일': 'C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\output\\RAG\\maple_inven_tips_embeddings.npy'}

,chunk_id,vector_first_8
0,inven_tip_48082_0,"[0.029084326699376106, -0.04240863397717476, -..."
1,inven_tip_48082_1,"[0.02551904134452343, -0.03824499621987343, -0..."
2,inven_tip_48082_2,"[0.018137749284505844, -0.03513288125395775, -..."
